# L4: Process Inputs - Chain of Thought Reasoning

In this section, we focus on **processing inputs**: tasks that take user input and generate useful output, often through multiple steps.

Sometimes models rush and make reasoning mistakes. A practical fix is to ask for a structured sequence of reasoning steps before the final answer. This approach is called **chain-of-thought reasoning**.

For some applications, reasoning details should not be shown to users. In those cases, use an **inner monologue** pattern: request structured intermediate reasoning, then display only the final user-facing portion.

## Setup
Load the API key and relevant Python libraries.

In [ ]:
import os
import openai
import sys
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
openai.api_key = os.environ['OPENAI_API_KEY']


def get_completion_from_messages(messages,
                                 model="gpt-3.5-turbo",
                                 temperature=0,
                                 max_tokens=500):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message["content"]

## Chain-of-Thought Prompting

This prompt asks the model to follow explicit steps:

1. Decide whether the query is about specific products.
2. Check whether those products are in the allowed product list.
3. Identify user assumptions.
4. Verify assumptions against product facts.
5. Politely correct incorrect assumptions and provide a friendly final response.

The delimiter format (`####`) makes each step easy to parse and helps separate intermediate reasoning from the final answer.

In [ ]:
delimiter = "####"
system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,\
i.e. {delimiter}.

Step 1:{delimiter} First decide whether the user is \
asking a question about a specific product or products. \
Product category doesn't count.

Step 2:{delimiter} If the user is asking about \
specific products, identify whether \
the products are in the following list.
All available products:
1. Product: TechPro Ultrabook
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-UB100
   Warranty: 1 year
   Rating: 4.5
   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor
   Description: A sleek and lightweight ultrabook for everyday use.
   Price: $799.99

2. Product: BlueWave Gaming Laptop
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-GL200
   Warranty: 2 years
   Rating: 4.7
   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060
   Description: A high-performance gaming laptop for an immersive experience.
   Price: $1199.99

3. Product: PowerLite Convertible
   Category: Computers and Laptops
   Brand: PowerLite
   Model Number: PL-CV300
   Warranty: 1 year
   Rating: 4.3
   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge
   Description: A versatile convertible laptop with a responsive touchscreen.
   Price: $699.99

4. Product: TechPro Desktop
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-DT500
   Warranty: 1 year
   Rating: 4.4
   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660
   Description: A powerful desktop computer for work and play.
   Price: $999.99

5. Product: BlueWave Chromebook
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-CB100
   Warranty: 1 year
   Rating: 4.1
   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS
   Description: A compact and affordable Chromebook for everyday tasks.
   Price: $249.99

Step 3:{delimiter} If the message contains products \
in the list above, list any assumptions that the \
user is making in their \
message e.g. that Laptop X is bigger than \
Laptop Y, or that Laptop Z has a 2 year warranty.

Step 4:{delimiter} If the user made any assumptions, \
figure out whether the assumption is true based on your \
product information.

Step 5:{delimiter} First, politely correct the \
customer's incorrect assumptions if applicable. \
Only mention or reference products in the list of \
5 available products, as these are the only 5 \
products that the store sells. \
Answer the customer in a friendly tone.

Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
"""

user_message = f"""
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop"""

messages = [
    {'role': 'system', 'content': system_message},
    {'role': 'user', 'content': f"{delimiter}{user_message}{delimiter}"},
]

response = get_completion_from_messages(messages)
print(response)

In [ ]:
user_message = f"""
do you sell tvs"""
messages = [
    {'role': 'system', 'content': system_message},
    {'role': 'user', 'content': f"{delimiter}{user_message}{delimiter}"},
]
response = get_completion_from_messages(messages)
print(response)

### What to observe from the two example queries

- Query 1 (`BlueWave Chromebook` vs `TechPro Desktop`) contains an incorrect assumption about which product is more expensive.
- A good response should detect and politely correct that assumption.
- Query 2 (`do you sell tvs`) is outside the listed inventory; the model may skip unnecessary intermediate steps and go straight to a direct user response.
- Prompt complexity is a trade-off: more steps can improve reliability but may be verbose. Test simpler prompts too and compare quality.

## Inner Monologue

Because reasoning steps are separated by a delimiter, we can hide chain-of-thought from end users and show only the final response.

This pattern is useful in cases like tutoring, where exposing full reasoning could reveal answers too directly.

The `try/except` extraction below is defensive: if the model output format is imperfect, your app still returns a safe fallback message.

In [ ]:
try:
    final_response = response.split(delimiter)[-1].strip()
except Exception as e:
    final_response = "Sorry, I'm having trouble right now, please try asking another question."

print(final_response)